                # Step 5 - Signature Extraction and Stability

                This notebook is the starting point for signature extraction. It uses a simple,
                leakage-safe repeated training-fold selection strategy so that the first stability
                analysis is already separated from baseline modeling.
                


In [ ]:
import sys; sys.path.insert(0, '../src')
                from collections import Counter

                import pandas as pd
                from sklearn.feature_selection import SelectKBest, f_classif
                from sklearn.model_selection import RepeatedStratifiedKFold

                from asd_pipeline_utils import RANDOM_STATE, get_targets, load_analysis_frame, split_features_and_metadata

                final_df = load_analysis_frame()
                X, meta = split_features_and_metadata(final_df)
                y_bin, _ = get_targets(meta)

                # Starter configuration for a first-pass binary signature screen.
                k_best = 100
                selector = SelectKBest(score_func=f_classif, k=k_best)
                cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)

                selection_counts = Counter()
                total_fits = 0

                for train_idx, _ in cv.split(X, y_bin):
                    X_train = X.iloc[train_idx]
                    y_train = y_bin.iloc[train_idx]
                    selector.fit(X_train, y_train)
                    selected_genes = X.columns[selector.get_support()].tolist()
                    selection_counts.update(selected_genes)
                    total_fits += 1

                stability_df = pd.DataFrame(
                    [
                        {
                            "gene": gene,
                            "selection_count": count,
                            "selection_frequency": count / total_fits,
                        }
                        for gene, count in selection_counts.items()
                    ]
                ).sort_values(["selection_count", "gene"], ascending=[False, True])

                output_path = "signature_stability_binary_top100.csv"
                stability_df.to_csv(output_path, index=False)
                print(f"Saved signature stability table to: {output_path}")
                stability_df.head(20)
                
